In [1]:
# Sistema de monitoreo agrícola de parcelas
# "Data Science en Recursos Naturales"
# Facultad de Ciencias Agronómicas, Universidad de Chile

In [30]:
import pandas as pd

In [31]:
def decision_riego(etc, pp):
    deficit = etc - pp
    if deficit <= 0:
        return "No regar"
    elif deficit < 4.0:
        return "Riego complementario"
    else:
        return "Riego urgente"

In [32]:
print("Decisión de riego:", decision_riego(5.2, 2.0))

Decisión de riego: Riego complementario


In [5]:
def dosis_nitrogeno(demanda, n_suelo, eficiencia=0.5):
    neto = demanda - n_suelo
    if neto <= 0:
        return 0
    return neto / eficiencia

In [6]:
print("Dosis N trigo:", dosis_nitrogeno(180, 45), "kg N/ha")

Dosis N trigo: 270.0 kg N/ha


In [7]:
# devuelve minimo, maximo y promedio
def estadisticas(valores):
    return min(valores), max(valores), sum(valores) / len(valores)

In [8]:
temp_min, temp_max, temp_prom = estadisticas([18, 21, 19, 25])
print("Mínimo:", temp_min)
print("Máximo:", temp_max)
print("Promedio:", temp_prom)

Mínimo: 18
Máximo: 25
Promedio: 20.75


In [9]:
# control de heladas durante la noche

In [10]:
temps = [8.5, 6.2, 4.1, 2.8, 1.5, 0.3, -0.8, -1.5, -0.2, 1.0]
horas = ["20:00", "21:00", "22:00", "23:00", "00:00",
         "01:00", "02:00", "03:00", "04:00", "05:00"]

In [11]:
horas_helada = 0
temp_min = temps[0]

In [12]:
for i in range(len(temps)):
    if temps[i] < temp_min:
        temp_min = temps[i]
    if temps[i] <= 0:
        horas_helada += 1
        print(f"[{horas[i]}] HELADA")
    elif temps[i] <= 3:
        print(f"[{horas[i]}] Precaución")

[23:00] Precaución
[00:00] Precaución
[01:00] Precaución
[02:00] HELADA
[03:00] HELADA
[04:00] HELADA
[05:00] Precaución


In [13]:
print("Temperatura mínima:", temp_min)
print("Horas de helada:", horas_helada)

Temperatura mínima: -1.5
Horas de helada: 3


In [14]:
# riego con while: se repite hasta llegar a 30% de humedad

In [15]:
humedad = 15
while humedad < 30:
    print(f"Regando... humedad: {humedad}%")
    humedad += 5
else:
    print("Riego completado")

Regando... humedad: 15%
Regando... humedad: 20%
Regando... humedad: 25%
Riego completado


In [16]:
# Lecturas de sensor con datos faltantes (None)
lecturas = [22.4, None, 21.8, None, 23.1]
for i, lectura in enumerate(lecturas):
    if lectura is None:
        continue
    print(f"Sensor {i}: {lectura} °C")

Sensor 0: 22.4 °C
Sensor 2: 21.8 °C
Sensor 4: 23.1 °C


In [17]:
# Clasificación de suelos (lista de diccionarios)
muestras = [
    {"sector": "Norte", "arena": 70, "arcilla": 15},
    {"sector": "Sur", "arena": 20, "arcilla": 45},
]

In [18]:
for m in muestras:
    if m["arena"] >= 70:
        textura = "Arenoso"
    elif m["arcilla"] >= 40:
        textura = "Arcilloso"
    elif m["arena"] >= 50:
        textura = "Franco-arenoso"
    else:
        textura = "Franco"
    print(f"{m['sector']}: {textura}")

Norte: Arenoso
Sur: Arcilloso


In [19]:
class Sensor:
    def __init__(self, id_sensor, tipo, valor):
        self.id_sensor = id_sensor
        self.tipo = tipo
        self._valor = valor

    def actualizar_valor(self, nuevo):
        if -50 <= nuevo <= 100:
            self._valor = nuevo
        else:
            print("Valor inválido")

    def evaluar(self):
        return "Sin evaluación"

    def __str__(self):
        return f"{self.id_sensor} ({self.tipo}): {self._valor}"


class SensorHumedad(Sensor):
    def __init__(self, id_sensor, valor):
        super().__init__(id_sensor, "humedad", valor)

    def evaluar(self):
        if self._valor < 15:
            return "Necesita riego"
        return "Humedad normal"

In [20]:
s = SensorHumedad("H-01", 12)
print(s)
print(s.evaluar())

s.actualizar_valor(38)
print(s, "->", s.evaluar())

H-01 (humedad): 12
Necesita riego
H-01 (humedad): 38 -> Humedad normal


In [21]:
class SensorTemperatura(Sensor):
    def __init__(self, id_sensor, valor):
        super().__init__(id_sensor, "temperatura", valor)

    def evaluar(self):
        if self._valor <= 0:
            return "Helada"
        return "Temperatura normal"


sensores = [SensorHumedad("H-01", 12), SensorTemperatura("T-01", -1.2)]

In [22]:
for s in sensores:
    print(s, "->", s.evaluar())

H-01 (humedad): 12 -> Necesita riego
T-01 (temperatura): -1.2 -> Helada


In [23]:
datos = {
    "parcela": ["A01", "A02", "A03", "A04", "A05", "A06", "A07", "A08"],
    "cultivo": ["Maíz", "Trigo", "Maíz", "Vid", "Trigo", "Vid", "Maíz", "Trigo"],
    "humedad_%": [24, 16, 13, 28, None, 12, 20, 18],
    "NDVI": [0.78, 0.49, 0.38, 0.65, 0.72, 0.44, None, 0.58],
    "rend_t_ha": [12.4, 6.8, 8.1, 10.2, 9.5, 5.9, 11.0, 7.7],
}
df = pd.DataFrame(datos)
df.to_csv("monitoreo_parcelas.csv", index=False)

In [24]:
df = pd.read_csv("monitoreo_parcelas.csv")
print(df.head())
print(df.shape)
df.info()

  parcela cultivo  humedad_%  NDVI  rend_t_ha
0     A01    Maíz       24.0  0.78       12.4
1     A02   Trigo       16.0  0.49        6.8
2     A03    Maíz       13.0  0.38        8.1
3     A04     Vid       28.0  0.65       10.2
4     A05   Trigo        NaN  0.72        9.5
(8, 5)
<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   parcela    8 non-null      str    
 1   cultivo    8 non-null      str    
 2   humedad_%  7 non-null      float64
 3   NDVI       7 non-null      float64
 4   rend_t_ha  8 non-null      float64
dtypes: float64(3), str(2)
memory usage: 328.0 bytes


In [25]:
# datos faltantes
print(df.isna().sum())
df["humedad_%"] = df["humedad_%"].fillna(df["humedad_%"].median())
df = df[df["NDVI"].notna()].copy()

parcela      0
cultivo      0
humedad_%    1
NDVI         1
rend_t_ha    0
dtype: int64


In [26]:
print(df.isna().sum())
df["humedad_%"] = df["humedad_%"].fillna(df["humedad_%"].median())
df = df[df["NDVI"].notna()].copy()

parcela      0
cultivo      0
humedad_%    0
NDVI         0
rend_t_ha    0
dtype: int64


In [27]:
# Variable nueva
df["clase_ndvi"] = df["NDVI"].apply(lambda x: "NDVI bajo" if x < 0.50 else "NDVI medio/alto")

In [28]:
# Resumen por cultivo
resumen = df.groupby("cultivo").agg(
    ndvi_medio=("NDVI", "mean"),
    rendimiento_medio=("rend_t_ha", "mean"),
    n_parcelas=("parcela", "count"),
).reset_index()
print(resumen)

  cultivo  ndvi_medio  rendimiento_medio  n_parcelas
0    Maíz    0.580000              10.25           2
1   Trigo    0.596667               8.00           3
2     Vid    0.545000               8.05           2


In [29]:
# CONCLUSIONES
# ---------------------------------------------------------
# 1. Heladas: durante la noche la temperatura mínima fue de -1.5 °C y hubo
#    3 horas de helada (02:00 a 04:00). Las horas anteriores y la última
#    quedaron en precaución (entre 0 y 3 °C). Se necesitaría activar el
#    control de heladas en esa franja.
#
# 2. Riego: con una ETc de 5.2 mm y 2.0 mm de precipitación, el déficit es
#    de 3.2 mm, por lo que corresponde riego complementario y no urgente.
#    En la simulación, el suelo pasó de 15 % a 30 % de humedad en 3 ciclos.
#
# 3. Fertilización: el trigo requiere 270 kg N/ha y el maíz 290.9 kg N/ha.
#    La vid no necesita fertilización nitrogenada, porque el nitrógeno del
#    suelo (90) supera su demanda (80).
#
# 4. Suelos: el sector Norte es arenoso (70 % de arena) y el Sur es
#    arcilloso (45 % de arcilla), por lo que tendrían un manejo de riego
#    distinto: el Norte retiene poca agua y el Sur drena lento.
#
# 5. Sensores: el sensor de humedad H-01 marcaba 12 % y necesitaba riego;
#    después de actualizarlo a 38 pasó a humedad normal. El sensor de
#    temperatura T-01 (-1.2 °C) indicó helada.
#
# 6. Parcelas: de las 8 parcelas, una se descartó por no tener NDVI.
#    De las 7 restantes, 3 tienen NDVI bajo (A02, A03 y A06) y 2 están en
#    alerta por tener NDVI y humedad bajos a la vez: A03 (Maíz) y A06 (Vid).
#    Esas dos serían las prioritarias para revisión en terreno.
#
# 7. Cultivos: el maíz tuvo el mayor rendimiento promedio (10.25 t/ha),
#    seguido de la vid (8.05) y el trigo (8.00). El trigo tuvo el NDVI
#    medio más alto (0.60) pero no el mayor rendimiento, así que el NDVI
#    por sí solo no explica el rendimiento.
#
# 8. Limitaciones: son pocas parcelas (2 o 3 por cultivo) y los datos son
#    de ejemplo, por lo que estos resultados no se pueden generalizar.
#    Los filtros solo identifican parcelas a revisar; no confirman la causa
#    del problema.